# Fine-tuning Qwen2.5-1.5B with LoRA on your Mac

Read this module's README first. It explains what we're building and why.

Close LM Studio (or unload its model) and any other memory-heavy apps before section 4.

In [ ]:
import gc
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from collections import Counter

import torch
import torch.nn as nn

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_DIR = "data"
ADAPTERS = "adapters"
FUSED = "models/darvis-router-1.5b"
FUSED_4BIT = "models/darvis-router-1.5b-4bit"
USE_WANDB = True

## 1. LoRA by hand

A linear layer computes `y = W x`. LoRA freezes `W` and adds a learned low-rank correction:

```
y = W x + (alpha / r) * B A x
```

`A` has shape `(r, d_in)` and `B` has shape `(d_out, r)`, with the rank `r` much smaller than either dimension. Their product `B A` is a full-size `(d_out, d_in)` matrix, but it's described by far fewer numbers. `B` starts at zero, so at step 0 the layer behaves exactly like the original model and training starts from the pretrained behavior instead of from noise.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                      # the pretrained weights stay frozen
        self.A = nn.Parameter(torch.randn(rank, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale


base = nn.Linear(4096, 4096)          # one projection matrix from a 7B-class model
lora = LoRALinear(base, rank=8)

frozen = sum(p.numel() for p in base.parameters())
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"frozen:    {frozen:>12,}")
print(f"trainable: {trainable:>12,}  ({trainable / frozen:.2%} of the layer)")

x = torch.randn(3, 4096)
print("identical to the base layer at init:", torch.allclose(lora(x), base(x)))

Why should a low-rank update be enough? The hypothesis behind LoRA (from the 2021 paper) is that the change fine-tuning needs to make has low "intrinsic rank": adapting a model to a task moves the weights in only a few important directions.

Test it on a toy problem. A frozen 64x64 layer needs to learn a hidden change that has rank exactly 4. LoRA with rank 4 should recover it. Rank 1 can't fully represent it.

In [ ]:
torch.manual_seed(0)
d = 64
W0 = nn.Linear(d, d, bias=False)
true_delta = 0.5 * torch.randn(d, 4) @ torch.randn(4, d) / math.sqrt(d)    # a rank-4 change

X = torch.randn(2048, d)
with torch.no_grad():
    Y = X @ (W0.weight + true_delta).T

for rank in [1, 4, 16]:
    torch.manual_seed(1)
    layer = LoRALinear(nn.Linear(d, d, bias=False), rank=rank, alpha=rank)
    layer.base.weight.data.copy_(W0.weight.data)
    opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
    for step in range(1500):
        loss = ((layer(X) - Y) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    learned = layer.scale * layer.B @ layer.A
    rel_err = ((learned - true_delta).norm() / true_delta.norm()).item()
    print(f"rank {rank:2}: final loss {loss.item():.6f}   relative error of learned update {rel_err:.3f}")

Rank 4 and up recover the change almost exactly. Rank 1 plateaus because it can only express one direction. In real fine-tuning nobody knows the true rank, so it's a hyperparameter: 8 to 16 is typical, and exercise 1 lets you test it.

MLX's implementation is the same idea. `mlx_lm` puts adapters on every linear layer (attention and MLP) in the last N transformer blocks.

## 2. Build the dataset

The task: map a student's question to `{"intent", "courses", "professors", "term"}`. Real Darvis logs would be the best training data. Without them, you generate synthetic data from templates, with the kinds of variation real users produce: nicknames for courses, inconsistent course codes, sloppy casing, optional terms.

The course list and professor names below are made up for the exercise. Swap in real ones from Darvis's database later.

In [ ]:
random.seed(0)

COURSES = {
    "CS 1114": ["intro to software design"],
    "CS 2114": ["software design and data structures", "software design"],
    "CS 2505": ["comp org 1", "computer organization 1", "comp org i"],
    "CS 2506": ["comp org 2", "computer organization 2", "comp org ii"],
    "CS 3114": ["data structures", "dsa", "data structures and algorithms"],
    "CS 3214": ["systems", "computer systems", "comp sys"],
    "CS 3304": ["comparative languages", "comp langs"],
    "CS 3714": ["mobile dev", "mobile software development"],
    "CS 4104": ["algorithms", "algo analysis", "data and algorithm analysis"],
    "CS 4824": ["machine learning", "the ml class"],
    "MATH 2114": ["linear algebra", "lin alg"],
    "MATH 2204": ["calc 3", "multivariable calc"],
    "STAT 3005": ["stats for engineers"],
}
PROFESSORS = ["Alvarez", "Chen", "Okafor", "Brennan", "Nakamura", "Rossi", "Haddad", "Kowalski", "Lindqvist", "Duarte"]
TERMS = {
    "Fall 2026": ["fall 2026", "Fall '26", "fall 26", "this fall"],
    "Spring 2027": ["spring 2027", "spring '27", "Spring 27", "next spring"],
    "Summer 2027": ["summer 2027", "summer '27"],
}

TEMPLATES = {
    "grade_distribution": [
        "what's the average gpa for {c}", "grade distribution for {c} with {p}",
        "how many people get an A in {c} under {p}", "is {c} with {p} a gpa booster",
        "what grades does {p} give in {c} {t}", "avg gpa {c} {t}", "how hard is it to get an A in {c}",
    ],
    "professor_compare": [
        "{p} or {p2} for {c}?", "who's better for {c}, {p} or {p2}", "compare {p} and {p2} in {c}",
        "is {p} easier than {p2} for {c} {t}", "{c}: {p} vs {p2}", "should i take {c} with {p} or {p2} {t}",
    ],
    "professor_recommend": [
        "who should i take for {c}", "best professor for {c} {t}", "which {c} prof is the easiest",
        "who teaches {c} best", "recommend a professor for {c} {t}",
    ],
    "prerequisites": [
        "what do i need before taking {c}", "prereqs for {c}?", "can i take {c} without {c2}",
        "is {c2} required for {c}", "what are the prerequisites of {c}",
    ],
    "course_info": [
        "what is {c} about", "how much work is {c}", "is {c} hard", "what topics does {c} cover",
        "should i take {c} and {c2} together {t}",
    ],
    "schedule": [
        "who is teaching {c} {t}", "is {c} offered {t}", "when is {c} usually offered", "is {p} teaching {c} {t}",
    ],
    "unsupported": [
        "what's the weather in blacksburg", "where's the best pizza near campus", "tell me a joke",
        "how do i reset my vt password", "who won the football game", "write my essay for me",
        "what time does the library close", "how do i get a parking pass", "when is spring break",
        "is the gym open today", "where can i print on campus", "help me find an internship",
        "what's the wifi password", "recommend a good laptop for college", "how do i join a club",
    ],
}


def course_mention(code):
    dept, num = code.split()
    if random.random() < 0.45:
        return random.choice(COURSES[code])
    return random.choice([f"{dept} {num}", f"{dept.lower()}{num}", f"{dept}{num}", f"{dept.lower()} {num}", f"{dept}-{num}"])


def prof_mention(name):
    return random.choice([name, name.lower(), f"Prof. {name}", f"professor {name}", f"Dr. {name}", f"dr {name.lower()}"])


def make_example():
    intent = random.choice(list(TEMPLATES))
    template = random.choice(TEMPLATES[intent])
    c, c2 = random.sample(list(COURSES), 2)
    p, p2 = random.sample(PROFESSORS, 2)

    term = None
    term_text = ""
    if "{t}" in template and random.random() < 0.6:
        term = random.choice(list(TERMS))
        term_text = random.choice(TERMS[term])

    question = template.format(c=course_mention(c), c2=course_mention(c2), p=prof_mention(p), p2=prof_mention(p2), t=term_text)
    if random.random() < 0.25:
        question = random.choice(["hey ", "quick question: ", "yo ", "darvis, ", "hi, "]) + question
    if random.random() < 0.25:
        question += random.choice(["?", " pls", " thanks", "??"])
    question = re.sub(r"\s+", " ", question).strip()
    if random.random() < 0.3:
        question = question.lower()

    def in_order(*pairs):   # keep entities in the order they appear in the template
        found = [(template.index(tag), value) for tag, value in pairs if tag in template]
        return [value for _, value in sorted(found)]

    target = {
        "intent": intent,
        "courses": in_order(("{c}", c), ("{c2}", c2)),
        "professors": in_order(("{p}", p), ("{p2}", p2)),
        "term": term,
    }
    return question, target


examples, seen = [], set()
while len(examples) < 900:
    q, t = make_example()
    if q.lower() not in seen:
        seen.add(q.lower())
        examples.append((q, t))

random.shuffle(examples)
train_ex, valid_ex, test_ex = examples[:700], examples[700:800], examples[800:]
print("test set intents:", dict(sorted(Counter(t["intent"] for _, t in test_ex).items())))
for q, t in train_ex[:6]:
    print(f"{q!r}\n   -> {json.dumps(t)}")

`mlx_lm` reads `train.jsonl`, `valid.jsonl`, and `test.jsonl` from a folder. Each line here is one conversation in the chat format from module 04: a short system prompt, the student's question, and the JSON answer we want.

In [ ]:
SYSTEM_SHORT = "You are the Darvis query router. Convert the student's question into a JSON query."

def to_chat(question, target):
    return {"messages": [
        {"role": "system", "content": SYSTEM_SHORT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": json.dumps(target)},
    ]}

os.makedirs(DATA_DIR, exist_ok=True)
for name, rows in [("train", train_ex), ("valid", valid_ex), ("test", test_ex)]:
    with open(f"{DATA_DIR}/{name}.jsonl", "w") as f:
        for q, t in rows:
            f.write(json.dumps(to_chat(q, t)) + "\n")
    print(f"{name}: {len(rows)} examples")

print()
print(open(f"{DATA_DIR}/train.jsonl").readline())

## 3. Baseline: the base model with its best prompt

To be fair to the base model, give it everything a prompt engineer would: the schema, the intents, the normalization rules, and the full course nickname table. This is the "just prompt it" alternative to fine-tuning.

In [ ]:
course_table = "\n".join(f"- {code}: {', '.join(aliases)}" for code, aliases in COURSES.items())
SYSTEM_DETAILED = f"""You are the Darvis query router for Virginia Tech students.
Convert the student's question into a JSON object with exactly these keys:
- "intent": one of "grade_distribution", "professor_compare", "professor_recommend", "prerequisites", "course_info", "schedule", "unsupported"
- "courses": list of course codes in the form "DEPT NUMBER" (for example "CS 3114"), in the order mentioned
- "professors": list of professor last names, capitalized, in the order mentioned
- "term": one of "Fall 2026", "Spring 2027", "Summer 2027", or null if no term is mentioned
Use "unsupported" with empty lists for questions that aren't about courses.
Course nicknames:
{course_table}
Reply with the JSON object only."""

print(SYSTEM_DETAILED)

In [ ]:
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
import mlx.core as mx

greedy = make_sampler(temp=0.0)

def predict(model, tokenizer, question, system):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return generate(model, tokenizer, prompt=prompt, max_tokens=100, sampler=greedy)


def parse(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def norm(values):
    return sorted(str(v).strip().lower() for v in (values or []))


def score(model, tokenizer, rows, system, label=""):
    fields = {"valid_json": 0, "intent": 0, "courses": 0, "professors": 0, "term": 0, "exact": 0}
    failures = []
    start = time.time()
    for q, target in rows:
        raw = predict(model, tokenizer, q, system)
        pred = parse(raw)
        if pred is None:
            failures.append((q, target, raw))
            continue
        fields["valid_json"] += 1
        ok = {
            "intent": pred.get("intent") == target["intent"],
            "courses": norm(pred.get("courses")) == norm(target["courses"]),
            "professors": norm(pred.get("professors")) == norm(target["professors"]),
            "term": pred.get("term") == target["term"],
        }
        for k, v in ok.items():
            fields[k] += v
        if all(ok.values()):
            fields["exact"] += 1
        else:
            failures.append((q, target, raw))
    n = len(rows)
    metrics = {k: v / n for k, v in fields.items()}
    print(f"{label:28} " + "  ".join(f"{k} {v:.0%}" for k, v in metrics.items()) + f"   ({time.time() - start:.0f}s)")
    return metrics, failures

In [ ]:
model, tokenizer = load(BASE_MODEL)      # downloads ~3 GB the first time

q, t = test_ex[0]
print("question:", q)
print("expected:", json.dumps(t))
print("base model says:", predict(model, tokenizer, q, SYSTEM_DETAILED))

In [ ]:
base_detailed, base_failures = score(model, tokenizer, test_ex, SYSTEM_DETAILED, "base + detailed prompt")
base_short, _ = score(model, tokenizer, test_ex, SYSTEM_SHORT, "base + one-line prompt")

print("\nSome failures with the detailed prompt:")
for q, t, raw in base_failures[:5]:
    print(f"  {q!r}\n    expected {json.dumps(t)}\n    got      {raw.strip()[:160]}")

Look at the kinds of mistakes: wrong intent labels, nicknames not mapped to codes, professor names with titles left in, extra prose around the JSON. A 1.5B model can follow a long prompt only so well.

Also count what that prompt costs. Every request pays for these tokens:

In [ ]:
n_detailed = len(tokenizer.encode(SYSTEM_DETAILED))
n_short = len(tokenizer.encode(SYSTEM_SHORT))
print(f"detailed system prompt: {n_detailed} tokens   one-line prompt: {n_short} tokens")

# Free the model before training starts in a separate process.
del model, tokenizer
gc.collect()
mx.clear_cache()

## 4. Fine-tune

The flags, in plain English:

| Flag | Meaning |
|---|---|
| `--train --data data` | Train on `data/train.jsonl`, checking `valid.jsonl` along the way |
| `--iters 600 --batch-size 4` | 600 optimizer steps of 4 examples each, about 3.4 passes over 700 examples |
| `--num-layers 16` | Put LoRA adapters in the last 16 of the model's 28 transformer blocks |
| `--learning-rate 2e-5` | Small, because MLX scales the adapter output by 20 by default |
| `--mask-prompt` | Only compute loss on the assistant's answer, not on the system prompt and question. You want the model to learn to produce the JSON, not to predict the question |
| `--steps-per-eval 100` | Report validation loss every 100 steps |
| `--adapter-path adapters` | Where to save the adapter weights |
| `--report-to wandb` | Send the loss curves to W&B |

The same command works in iTerm2 from this folder: `uv run mlx_lm.lora --model Qwen/Qwen2.5-1.5B-Instruct --train ...`.

In [ ]:
def run(cmd):
    print("$", " ".join(cmd), "\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("command failed, see the output above")

train_cmd = [
    sys.executable, "-m", "mlx_lm", "lora",
    "--model", BASE_MODEL,
    "--train", "--data", DATA_DIR,
    "--iters", "600", "--batch-size", "4",
    "--num-layers", "16",
    "--learning-rate", "2e-5",
    "--mask-prompt",
    "--steps-per-report", "20", "--steps-per-eval", "100", "--val-batches", "25",
    "--save-every", "200",
    "--adapter-path", ADAPTERS,
]
if USE_WANDB:
    train_cmd += ["--report-to", "wandb", "--project-name", "ml-lab-lora"]

start = time.time()
run(train_cmd)
print(f"\ntraining took {(time.time() - start) / 60:.1f} minutes")

Reading the log:

- "Trainable parameters" near the top shows the percentage of weights LoRA is training. Compare it with section 1.
- Train loss should fall quickly, because the output format is very regular.
- Validation loss is the number to trust. If it starts rising while train loss keeps falling, the adapter is memorizing (the overfitting pattern from module 05).
- "Peak mem" tells you how much headroom you have for a bigger model or batch.

In [ ]:
print(sorted(os.listdir(ADAPTERS)))
size_mb = sum(os.path.getsize(os.path.join(ADAPTERS, f)) for f in os.listdir(ADAPTERS)) / 1e6
print(f"adapter folder: {size_mb:.0f} MB (the base model is about 3,000 MB)")

## 5. Did it work?

Load the base model with the adapter attached and score it on the same test set, now with only the one-line prompt.

In [ ]:
ft_model, ft_tokenizer = load(BASE_MODEL, adapter_path=ADAPTERS)
ft_short, ft_failures = score(ft_model, ft_tokenizer, test_ex, SYSTEM_SHORT, "fine-tuned + one-line prompt")

print()
print(f"{'':28} {'exact match':>12} {'prompt tokens':>14}")
print(f"{'base + detailed prompt':28} {base_detailed['exact']:12.0%} {n_detailed:14}")
print(f"{'base + one-line prompt':28} {base_short['exact']:12.0%} {n_short:14}")
print(f"{'fine-tuned + one-line prompt':28} {ft_short['exact']:12.0%} {n_short:14}")

In [ ]:
print("Remaining failures:")
for q, t, raw in ft_failures[:8]:
    print(f"  {q!r}\n    expected {json.dumps(t)}\n    got      {raw.strip()[:160]}")

Try your own questions, including ones phrased differently from the templates:

In [ ]:
for q in [
    "yo is chen chill for systems this fall",
    "which prof for lin alg gives the most As",
    "do i need comp org 2 before taking cs3214?",
    "what's the dining hall menu today",
]:
    print(f"{q!r}\n  -> {predict(ft_model, ft_tokenizer, q, SYSTEM_SHORT).strip()}")

Two honest caveats about these numbers:

1. The test set comes from the same templates as the training set, so it measures whether the model learned the task as defined, not how it handles truly new phrasings. Exercise 5 probes that.
2. Some "failures" may be ambiguous questions where the label itself is debatable. Read them. Error analysis is where most of the real improvement in applied ML comes from.

## 6. Fuse and quantize

For serving, merge the adapter into the base weights so you have one ordinary model folder, then make a 4-bit copy that's smaller and faster.

In [ ]:
del ft_model, ft_tokenizer
gc.collect()
mx.clear_cache()

os.makedirs("models", exist_ok=True)
run([sys.executable, "-m", "mlx_lm", "fuse", "--model", BASE_MODEL, "--adapter-path", ADAPTERS, "--save-path", FUSED])

if os.path.exists(FUSED_4BIT):
    print(f"{FUSED_4BIT} already exists. Delete it to rebuild.")
else:
    run([sys.executable, "-m", "mlx_lm", "convert", "--hf-path", FUSED, "--mlx-path", FUSED_4BIT, "-q"])

def folder_gb(path):
    return sum(os.path.getsize(os.path.join(dp, f)) for dp, _, files in os.walk(path) for f in files) / 1e9

print(f"\nfused 16-bit: {folder_gb(FUSED):.2f} GB   fused 4-bit: {folder_gb(FUSED_4BIT):.2f} GB")

In [ ]:
q_model, q_tokenizer = load(FUSED_4BIT)
q_short, _ = score(q_model, q_tokenizer, test_ex, SYSTEM_SHORT, "fused 4-bit + one-line prompt")

Compare the 4-bit score with the 16-bit fine-tuned score. A small drop is normal. A large one means the task is sensitive to precision, and you'd serve the 8-bit or 16-bit version instead (`--q-bits 8` on the convert command).

`models/darvis-router-1.5b-4bit` is what module 08 serves.

To keep the model, push it to a private Hub repo instead of committing it:

```python
from huggingface_hub import HfApi
api = HfApi()
repo = f"{api.whoami()['name']}/darvis-router-1.5b-4bit"
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=FUSED_4BIT, repo_id=repo)
```

Qwen2.5-1.5B-Instruct is Apache 2.0 licensed, so you can use fine-tunes of it commercially. Check the license every time you pick a base model.

---
# Exercises

**Exercise 1: Rank.** `mlx_lm lora` accepts a YAML config file through `--config`. Write `rank16.yaml` with `lora_parameters: {rank: 16, dropout: 0.0, scale: 20.0}`, retrain into `adapters_r16/`, and compare exact match and adapter size with the default (rank 8). Try rank 2 as well. Command-line flags override the YAML, so keep your other flags the same.

**Exercise 2: How much data do you need?** Write `data_small/` containing only the first 100 training examples (plus the same `valid.jsonl`), train for 200 iterations into `adapters_small/`, and score it. Then try 300 examples. Sketch the accuracy-vs-data curve. This is the question you'll face with real Darvis logs: how many labeled examples are worth collecting?

**Exercise 3: The cost argument.** You measured prompt tokens for the detailed and one-line prompts. If Darvis handled 100,000 routing requests a month on a hosted model priced at `IN_PRICE` dollars per million input tokens, how much does the detailed prompt alone cost per month compared with the one-line prompt? Include the question tokens (estimate the average from `test_ex`).

**Exercise 4: Error analysis.** Group the fine-tuned model's failures by the expected intent and by which field was wrong. Pick the most common failure type, write a few new templates or nicknames that cover it, regenerate the data, retrain, and check whether that failure type dropped.

**Exercise 5: Out-of-distribution test.** Write 15 questions by hand the way a real student would type them: slang, typos, two questions in one, a course that isn't in the list. Score the fine-tuned model on them. Where does synthetic training data break down?

---
# Solutions

In [ ]:
# Exercise 1
with open("rank16.yaml", "w") as f:
    f.write("lora_parameters:\n  rank: 16\n  dropout: 0.0\n  scale: 20.0\n")

r16_cmd = [c if c != ADAPTERS else "adapters_r16" for c in train_cmd] + ["--config", "rank16.yaml"]
run(r16_cmd)

m, tkn = load(BASE_MODEL, adapter_path="adapters_r16")
score(m, tkn, test_ex, SYSTEM_SHORT, "rank 16")
size = sum(os.path.getsize(os.path.join("adapters_r16", f)) for f in os.listdir("adapters_r16")) / 1e6
print(f"adapter folder: {size:.0f} MB (includes checkpoints saved every 200 steps)")
del m, tkn; gc.collect(); mx.clear_cache()

In [ ]:
# Exercise 2
os.makedirs("data_small", exist_ok=True)
with open(f"{DATA_DIR}/train.jsonl") as f:
    lines = f.readlines()[:100]
with open("data_small/train.jsonl", "w") as f:
    f.writelines(lines)
with open(f"{DATA_DIR}/valid.jsonl") as src, open("data_small/valid.jsonl", "w") as dst:
    dst.write(src.read())

small_cmd = [c for c in train_cmd]
small_cmd[small_cmd.index("--data") + 1] = "data_small"
small_cmd[small_cmd.index("--iters") + 1] = "200"
small_cmd[small_cmd.index("--adapter-path") + 1] = "adapters_small"
run(small_cmd)

m, tkn = load(BASE_MODEL, adapter_path="adapters_small")
score(m, tkn, test_ex, SYSTEM_SHORT, "100 training examples")
del m, tkn; gc.collect(); mx.clear_cache()

In [ ]:
# Exercise 3
from transformers import AutoTokenizer
tok_only = AutoTokenizer.from_pretrained(BASE_MODEL)   # already in your cache from section 3
avg_question = sum(len(tok_only.encode(q)) for q, _ in test_ex) / len(test_ex)
template_overhead = 15            # rough tokens added by the chat template markers

IN_PRICE = 0.50                   # placeholder $ per 1M input tokens. Replace with a real price.
requests = 100_000
for name, n_sys in [("detailed", n_detailed), ("one-line", n_short)]:
    tokens = n_sys + avg_question + template_overhead
    print(f"{name:9} {tokens:6.0f} input tokens/request -> ${requests * tokens * IN_PRICE / 1e6:,.2f}/month")

In [ ]:
# Exercise 4
by_intent, by_field = Counter(), Counter()
for q, t, raw in ft_failures:
    by_intent[t["intent"]] += 1
    pred = parse(raw)
    if pred is None:
        by_field["invalid json"] += 1
        continue
    if pred.get("intent") != t["intent"]: by_field["intent"] += 1
    if norm(pred.get("courses")) != norm(t["courses"]): by_field["courses"] += 1
    if norm(pred.get("professors")) != norm(t["professors"]): by_field["professors"] += 1
    if pred.get("term") != t["term"]: by_field["term"] += 1

print("failures by expected intent:", by_intent.most_common())
print("failures by wrong field:    ", by_field.most_common())
# Next: add templates/nicknames targeting the top entry, rerun section 2, retrain, rescore.

In [ ]:
# Exercise 5
ood = [
    ("is alvarez good for dsa or should i wait till spring '27",
     {"intent": "professor_recommend", "courses": ["CS 3114"], "professors": ["Alvarez"], "term": "Spring 2027"}),
    ("whats the curve like in chens systems class",
     {"intent": "grade_distribution", "courses": ["CS 3214"], "professors": ["Chen"], "term": None}),
    ("prereqs for ml?? also who teaches it",
     {"intent": "prerequisites", "courses": ["CS 4824"], "professors": [], "term": None}),
    ("is CS 4644 hard",
     {"intent": "course_info", "courses": ["CS 4644"], "professors": [], "term": None}),
    ("rossi vs okafor vs nakamura for algorithms",
     {"intent": "professor_compare", "courses": ["CS 4104"], "professors": ["Rossi", "Okafor", "Nakamura"], "term": None}),
]
# Add 10 more of your own, then:
m, tkn = load(FUSED_4BIT)
_, ood_failures = score(m, tkn, ood, SYSTEM_SHORT, "out-of-distribution")
for q, t, raw in ood_failures:
    print(f"  {q!r}\n    expected {json.dumps(t)}\n    got      {raw.strip()[:160]}")